# BÀI TẬP SINH VIÊN — TRANSFER LEARNING VỚI EFFICIENTNET-B0 TRÊN FGVC-AIRCRAFT

**Đề bài:** Huấn luyện mô hình cho bộ dữ liệu **FGVC-Aircraft** sử dụng mô hình pretrained **EfficientNet-B0** (ImageNet-1K).

**Môi trường:** Google Colab GPU. Chọn **Runtime → Change runtime type → T4 GPU**, sau đó **Runtime → Run all**.

**Quy trình giữ nguyên như Bài 6 (Flowers-102 + ResNet18):** 4 chiến lược Scratch / Linear Probe / Partial FT / Full FT → chọn mô hình **chỉ bằng validation** → đánh giá test **đúng một lần** → phân tích lỗi → lưu/tải checkpoint.

### Kết quả học tập
1. Áp dụng pretrained **EfficientNet-B0**: thay `classifier`, đóng băng khối, quản lý BatchNorm, learning rate theo nhóm khối.
2. Làm việc với bài toán **fine-grained** (phân biệt các biến thể máy bay rất giống nhau) và nhận ra khác biệt với Flowers-102.
3. So sánh 4 chiến lược trên validation theo loss / top-1 / macro-F1 / top-5.
4. Đánh giá test một lần, phân tích lớp khó, cặp lớp hay nhầm, ảnh sai tự tin cao.
5. Lưu checkpoint, tải lại và suy luận.

**Dữ liệu:** https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/  
**PyTorch transfer learning:** https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html  
**EfficientNet-B0 (torchvision):** https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b0.html

| Phần | Nội dung | Kết quả |
|---|---|---|
| 0–2 | Thiết lập, dữ liệu, tiền xử lý | Kiểm tra split / tensor / ảnh mẫu |
| 3–4 | Kiến trúc EfficientNet-B0, hàm train/evaluate | Bảng số tham số |
| 5–8 | Scratch, Linear Probe, Partial FT, Full FT | Learning curves, bảng validation |
| 9–11 | Chọn mô hình, test một lần, phân tích lỗi | Bảng test, confusion, ảnh sai |
| 12–13 | Checkpoint, suy luận, nhận xét, mở rộng | File `.pt`, kết luận |

> **Quy tắc khoa học:** không xem test khi chỉnh mô hình. Mọi nhận xét chỉ là quan sát trên **một lần chạy (một seed)**, không phải kết luận thống kê.

## 0. Thiết lập và kiểm soát tái lập

- Dùng GPU CUDA, **seed = 42**.
- Số epoch mặc định: Scratch / Partial / Full = **10**, Linear Probe = **40** (chạy trên embedding đã cache nên rất nhanh). Có thể chỉnh ở các hằng số `EPOCHS_*`.
- Dữ liệu (~2,75 GB) tải trực tiếp từ website Oxford qua `torchvision.datasets.FGVCAircraft`, **lần đầu mất vài phút**.

In [ ]:
!nvidia-smi -L

In [ ]:
import copy
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models, transforms
from torchvision.models import EfficientNet_B0_Weights

SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = min(4, os.cpu_count() or 2)
IMAGE_SIZE = 224
ANNOTATION_LEVEL = "variant"   # 'variant' = 100 lớp (khó nhất), 'family' = 70, 'manufacturer' = 30
EPOCHS_SCRATCH = 10
EPOCHS_PROBE = 40
EPOCHS_PARTIAL = 10
EPOCHS_FULL = 10
PATIENCE = 5
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./bai_tap_aircraft_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fallback to CPU instead of strict assertion failure, with a warning
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Đang sử dụng GPU CUDA.")
else:
    DEVICE = torch.device("cpu")
    print("CẢNH BÁO: Không tìm thấy GPU CUDA. Đã chuyển sang sử dụng CPU (sẽ chạy rất chậm).")
    print("Để bật GPU: Runtime → Change runtime type → Chọn T4 GPU.")

WEIGHTS = EfficientNet_B0_Weights.IMAGENET1K_V1


def set_seed(seed: int = SEED) -> None:
    """Đặt seed cho các thư viện."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
print("Torch:", torch.__version__, "| Torchvision:", __import__("torchvision").__version__)
print("Thiết bị:", DEVICE, "| Seed:", SEED, "| Workers:", NUM_WORKERS)

## 1. FGVC-Aircraft: bài toán **fine-grained**

FGVC-Aircraft gồm **10.000 ảnh máy bay**, chia chính thức **train 3.334 / val 3.333 / test 3.333**. Mỗi ảnh có 3 mức nhãn phân cấp:

| Mức nhãn | Số lớp | Ví dụ |
|---|---|---|
| `manufacturer` | 30 | Boeing, Airbus |
| `family` | 70 | Boeing 737 |
| `variant` | **100** | Boeing 737-700 |

Bài này dùng mức **`variant` (100 lớp)**: mỗi lớp khoảng 33 ảnh train. Khác với Flowers-102 (phân biệt loài hoa khác nhau rõ về màu/hình), các lớp ở đây là **các biến thể của cùng một dòng máy bay**, khác nhau ở chi tiết nhỏ (số cửa sổ, số động cơ, hình dáng đuôi...). Vì vậy:

- Đặc trưng ImageNet có thể **chưa đủ** → fine-tuning dự kiến có lợi hơn linear probe (so với Flowers-102). Đây là **giả thuyết cần kiểm chứng**, không phải kết luận có sẵn.
- Mỗi ảnh có **dải bản quyền cao 20 px ở cạnh dưới** → cần cắt bỏ để mô hình không học chữ.

| Khái niệm | Trong bài |
|---|---|
| Source domain / task | ImageNet-1K, phân loại 1.000 lớp |
| Target domain / task | Ảnh máy bay, phân loại 100 biến thể |
| Backbone | `model.features` của EfficientNet-B0 (đầu ra 1.280 kênh) |
| Head | `Linear(1280, 100)` thay `Linear(1280, 1000)` trong `model.classifier` |

## 2. Tiền xử lý, Dataset và DataLoader

- Chuẩn hóa ImageNet: `mean=(0.485, 0.456, 0.406)`, `std=(0.229, 0.224, 0.225)`.
- **Cắt 20 px dưới cùng** (dải bản quyền) trước mọi phép biến đổi khác.
- **Eval/Val/Test:** `Resize((224, 224))` (không `CenterCrop`, vì ảnh máy bay rộng ~3:2, crop giữa có thể cắt mất mũi/đuôi, đúng phần giúp phân biệt biến thể).
- **Train:** `RandomResizedCrop(224, scale=(0.6, 1.0), ratio=(1.2, 1.8))`, lật ngang, ColorJitter nhẹ.
- **Linear probe cache:** train dùng biến đổi deterministic để mỗi ảnh có đúng một embedding.
- Không lấy thống kê từ tập test; không lật dọc.

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


class CropBanner:
    """Cắt dải bản quyền cao 20 px ở đáy ảnh FGVC-Aircraft."""
    def __init__(self, pixels: int = 20):
        self.pixels = pixels

    def __call__(self, img):
        width, height = img.size
        return img.crop((0, 0, width, height - self.pixels))


train_transform = transforms.Compose([
    CropBanner(),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0), ratio=(1.2, 1.8)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    CropBanner(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

aircraft_train = datasets.FGVCAircraft(DATA_DIR, split="train", annotation_level=ANNOTATION_LEVEL,
                                       transform=train_transform, download=True)
aircraft_train_clean = datasets.FGVCAircraft(DATA_DIR, split="train", annotation_level=ANNOTATION_LEVEL,
                                             transform=eval_transform, download=False)
aircraft_val = datasets.FGVCAircraft(DATA_DIR, split="val", annotation_level=ANNOTATION_LEVEL,
                                     transform=eval_transform, download=False)
aircraft_test = datasets.FGVCAircraft(DATA_DIR, split="test", annotation_level=ANNOTATION_LEVEL,
                                      transform=eval_transform, download=False)

CLASS_NAMES = list(aircraft_train.classes)
NUM_CLASSES = len(CLASS_NAMES)
print("Số lớp:", NUM_CLASSES, "| Ví dụ:", CLASS_NAMES[:4])
print("Train:", len(aircraft_train), "| Val:", len(aircraft_val), "| Test:", len(aircraft_test))
assert NUM_CLASSES == 100
assert len(aircraft_train) == len(aircraft_train_clean)
assert len(aircraft_val) > 0 and len(aircraft_test) > 0

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(aircraft_train, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
train_clean_loader = DataLoader(aircraft_train_clean, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
val_loader = DataLoader(aircraft_val, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

sample_images, sample_labels = next(iter(train_loader))
print("Một batch:", tuple(sample_images.shape), "| Nhãn:", tuple(sample_labels.shape))
assert sample_images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)

counts = np.bincount([label for label in aircraft_train._labels], minlength=NUM_CLASSES)
print(f"Ảnh/lớp ở train: min={counts.min()}, max={counts.max()}, trung bình={counts.mean():.1f}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, idx in zip(axes.flat, np.linspace(0, len(aircraft_train_clean) - 1, 8, dtype=int)):
    img, target = aircraft_train_clean[int(idx)]
    arr = img.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)
    ax.imshow(np.clip(arr, 0.0, 1.0))
    ax.set_title(CLASS_NAMES[target], fontsize=10)
    ax.axis("off")
fig.suptitle("FGVC-Aircraft: ví dụ từ train (đã cắt dải bản quyền, chưa có ảnh test)")
plt.tight_layout(); plt.show()

### Hướng dẫn đọc kết quả — Dữ liệu
Kỳ vọng: `Số lớp: 100`, train/val/test lần lượt khoảng **3.334 / 3.333 / 3.333** ảnh, batch `(32, 3, 224, 224)`. Ảnh/lớp ở train xấp xỉ 33 (ít hơn nhiều so với kích thước mô hình, nhưng nhiều hơn 10 ảnh/lớp của Flowers-102). Hãy quan sát ảnh mẫu: nhiều biến thể của cùng hãng nhìn rất giống nhau.

## 3. Khám phá EfficientNet-B0: backbone, head và số tham số

EfficientNet-B0 trong torchvision gồm:
- `model.features`: 9 khối tuần tự `features[0]` (stem) … `features[8]` (conv 1×1 → 1.280 kênh); `features[1..7]` là các tầng MBConv.
- `model.avgpool` rồi `model.classifier = Sequential(Dropout(0.2), Linear(1280, 1000))`.

Ta thay `model.classifier[1]` bằng `Linear(1280, 100)`.

Bốn chiến lược:

| Chiến lược | Tham số học được |
|---|---|
| **scratch** | Toàn bộ mạng, `weights=None` |
| **probe** | Chỉ head tuyến tính trên embedding cache |
| **partial** | `features[6:]` (3 khối cuối) + `classifier` |
| **full** | Toàn mạng, learning rate tăng dần theo độ sâu |

**BatchNorm:** `requires_grad=False` không khóa `running_mean/var`. Với probe/partial, ta đặt mọi BatchNorm ở `eval()` trong lúc train.

In [ ]:
def count_parameters(model: nn.Module) -> dict:
    """Đếm tổng số tham số và số tham số được phép học."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"Total": total, "Trainable": trainable, "Frozen": total - trainable}


def build_efficientnet_b0(num_classes: int = NUM_CLASSES, pretrained: bool = True) -> nn.Module:
    """Khởi tạo EfficientNet-B0 và thay head theo số lớp đích."""
    model = models.efficientnet_b0(weights=WEIGHTS if pretrained else None)
    in_features = model.classifier[1].in_features   # 1280
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


PARTIAL_FROM = 6   # mở features[6], features[7], features[8]


def configure_trainable(model: nn.Module, strategy: str) -> nn.Module:
    """Chọn phần trainable: scratch, probe, partial, full."""
    if strategy not in ("scratch", "probe", "partial", "full"):
        raise ValueError(f"Chiến lược không được hỗ trợ: {strategy}")
    for parameter in model.parameters():
        parameter.requires_grad = strategy in ("scratch", "full")
    if strategy in ("probe", "partial"):
        for parameter in model.classifier.parameters():
            parameter.requires_grad = True
    if strategy == "partial":
        for block in model.features[PARTIAL_FROM:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
    return model


def set_training_mode(model: nn.Module, strategy: str) -> None:
    """Phân biệt train() với requires_grad; giữ BatchNorm ở eval với probe/partial."""
    model.train()
    if strategy in ("probe", "partial"):
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()


class LinearProbe(nn.Module):
    """Head tuyến tính cho embedding 1280 chiều đã cache."""
    def __init__(self, in_features: int = 1280, out_features: int = NUM_CLASSES):
        super().__init__()
        self.classifier = nn.Linear(in_features, out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)


def build_optimizer(model: nn.Module, strategy: str) -> torch.optim.Optimizer:
    """Head học nhanh hơn backbone; tham số đóng băng không đưa vào optimizer."""
    if strategy == "scratch":
        return torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    if strategy == "probe":
        return torch.optim.AdamW(model.classifier.parameters(), lr=3e-3, weight_decay=1e-3)
    if strategy == "partial":
        return torch.optim.AdamW([
            {"params": model.features[PARTIAL_FROM:].parameters(), "lr": 3e-4},
            {"params": model.classifier.parameters(), "lr": 1e-3},
        ], weight_decay=1e-4)
    if strategy == "full":
        return torch.optim.AdamW([
            {"params": model.features[:4].parameters(), "lr": 5e-5},
            {"params": model.features[4:PARTIAL_FROM].parameters(), "lr": 1e-4},
            {"params": model.features[PARTIAL_FROM:].parameters(), "lr": 3e-4},
            {"params": model.classifier.parameters(), "lr": 1e-3},
        ], weight_decay=1e-4)
    raise ValueError(strategy)


shape_demo = build_efficientnet_b0(num_classes=NUM_CLASSES, pretrained=False)
probe_demo = configure_trainable(copy.deepcopy(shape_demo), "probe")
partial_demo = configure_trainable(copy.deepcopy(shape_demo), "partial")
full_demo = configure_trainable(copy.deepcopy(shape_demo), "full")
display(pd.DataFrame([
    {"Chiến lược": "Scratch", **count_parameters(shape_demo)},
    {"Chiến lược": "Frozen + head", **count_parameters(probe_demo)},
    {"Chiến lược": "features[6:] + head", **count_parameters(partial_demo)},
    {"Chiến lược": "All layers", **count_parameters(full_demo)},
]))
assert probe_demo.classifier[1].out_features == NUM_CLASSES
assert 0 < count_parameters(probe_demo)["Trainable"] < count_parameters(partial_demo)["Trainable"]
assert count_parameters(partial_demo)["Trainable"] < count_parameters(full_demo)["Trainable"]
del shape_demo, probe_demo, partial_demo, full_demo

### Hướng dẫn đọc kết quả — Số tham số
**Total** bằng nhau ở cả bốn chiến lược; **Trainable** là phần optimizer thực sự cập nhật. Head `Linear(1280, 100)` có `1280×100 + 100 = 128.100` tham số. EfficientNet-B0 nhỏ hơn ResNet18 nhiều (~4 triệu so với ~11 triệu tham số) nhưng sâu hơn và dùng depthwise convolution.

## 4. Hàm train / evaluate tái sử dụng

Quy trình: `zero_grad → forward → CrossEntropyLoss(logits, labels) → backward → step`.

- `CrossEntropyLoss` nhận **logits** (không softmax trước).
- Loss tổng hợp **theo số mẫu**, không trung bình các trung bình batch.
- **Checkpoint tốt nhất chọn bằng `val_loss`** cho mọi phương pháp; early stopping với `PATIENCE`.
- `evaluate` trả thêm **độ tự tin** (xác suất lớp dự đoán) để phân tích ảnh sai tự tin cao.

In [ ]:
def evaluate(model: nn.Module, loader: DataLoader, num_classes: int = NUM_CLASSES) -> dict:
    """Đánh giá theo mẫu: loss, top-1, top-5, macro-F1, nhãn, dự đoán, độ tự tin."""
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    total_loss, total, correct_1, correct_5 = 0.0, 0, 0, 0
    all_targets, all_preds, all_conf = [], [], []
    with torch.inference_mode():
        for inputs, targets in loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            logits = model(inputs)
            total_loss += criterion(logits, targets).item()
            probs = logits.softmax(dim=1)
            conf, preds = probs.max(dim=1)
            correct_1 += int((preds == targets).sum())
            correct_5 += int((logits.topk(min(5, num_classes), dim=1).indices ==
                              targets.unsqueeze(1)).any(dim=1).sum())
            total += targets.size(0)
            all_targets.extend(targets.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_conf.extend(conf.cpu().tolist())
    assert total > 0
    return {"loss": total_loss / total, "accuracy": correct_1 / total, "top5": correct_5 / total,
            "macro_f1": f1_score(all_targets, all_preds, labels=list(range(num_classes)),
                                 average="macro", zero_division=0),
            "targets": np.asarray(all_targets), "preds": np.asarray(all_preds),
            "conf": np.asarray(all_conf)}


def fit_model(name: str, model: nn.Module, train_data: DataLoader, val_data: DataLoader,
              strategy: str, epochs: int, num_classes: int = NUM_CLASSES):
    """Huấn luyện, in từng epoch, giữ best checkpoint theo validation loss."""
    model = model.to(DEVICE)
    optimizer = build_optimizer(model, strategy)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    criterion = nn.CrossEntropyLoss()
    history, best_state = [], None
    best_loss, best_epoch, patience_count = float("inf"), 0, 0
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        set_training_mode(model, strategy)
        total_loss, num_seen = 0.0, 0
        for inputs, targets in train_data:
            inputs = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(targets)
            num_seen += len(targets)
        val_metrics = evaluate(model, val_data, num_classes)
        record = {"epoch": epoch, "train_loss": total_loss / num_seen, "val_loss": val_metrics["loss"],
                  "val_acc": val_metrics["accuracy"], "val_f1": val_metrics["macro_f1"],
                  "val_top5": val_metrics["top5"]}
        history.append(record)
        print(f"[{name}] {epoch:02d}/{epochs:02d} | train_loss={record['train_loss']:.4f} "
              f"| val_loss={record['val_loss']:.4f} | val_acc={record['val_acc']:.2%} "
              f"| val_macroF1={record['val_f1']:.4f}")
        if val_metrics["loss"] < best_loss - 1e-6:
            best_loss, best_epoch, patience_count = val_metrics["loss"], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
        scheduler.step()
        if patience_count >= PATIENCE:
            print(f"[{name}] Early stopping tại epoch {epoch}.")
            break
    if best_state is None:
        raise RuntimeError("Không có checkpoint hợp lệ")
    model.load_state_dict(best_state)
    metrics = evaluate(model, val_data, num_classes)
    seconds = time.perf_counter() - start
    summary = {"Mô hình": name, "Best epoch": best_epoch, "Val loss": metrics["loss"],
               "Val accuracy": metrics["accuracy"], "Val macro-F1": metrics["macro_f1"],
               "Val top-5": metrics["top5"], "Thời gian (s)": round(seconds, 1),
               "Trainable": count_parameters(model)["Trainable"]}
    print(f"[{name}] BEST epoch={best_epoch}, val_loss={metrics['loss']:.4f}, "
          f"val_accuracy={metrics['accuracy']:.2%}")
    return model, pd.DataFrame(history), summary


def plot_histories(histories: dict, title: str) -> None:
    """Vẽ validation accuracy và train/val loss của các phương pháp."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for name, frame in histories.items():
        axes[0].plot(frame.epoch, frame.val_acc, marker="o", label=name)
        axes[1].plot(frame.epoch, frame.train_loss, linestyle="--", alpha=0.5, label=f"{name} train")
        axes[1].plot(frame.epoch, frame.val_loss, marker="o", label=f"{name} val")
    axes[0].set(xlabel="Epoch", ylabel="Accuracy", title="Validation accuracy", ylim=(0, 1))
    axes[1].set(xlabel="Epoch", ylabel="Cross-entropy", title="Train loss / Val loss")
    for ax in axes:
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

## 5. Thí nghiệm A — EfficientNet-B0 **huấn luyện từ đầu** (baseline)

`weights=None`, tối ưu toàn mạng. **Giả thuyết:** với ~33 ảnh/lớp và ngân sách ngắn, mô hình từ đầu khó học đặc trưng fine-grained; 10 epoch **chưa đủ để hội tụ**. Accuracy ngẫu nhiên là 1%.

In [ ]:
histories, fitted_models, summaries = {}, {}, {}
set_seed(SEED)
scratch_model = configure_trainable(build_efficientnet_b0(pretrained=False), "scratch")
scratch_model, histories["Scratch"], summaries["Scratch"] = fit_model(
    "Scratch", scratch_model, train_loader, val_loader, strategy="scratch", epochs=EPOCHS_SCRATCH)
fitted_models["Scratch"] = scratch_model.cpu()
display(pd.DataFrame([summaries["Scratch"]]).round(4))
del scratch_model
torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Scratch] 01/10 | train_loss=4.7034 | val_loss=4.5511 | val_acc=1.86% | val_macroF1=0.0046


## 6. Thí nghiệm B — **Frozen feature extractor + linear probing**

Giữ nguyên backbone ImageNet, trích vector **1.280 chiều** một lần cho mỗi ảnh (train dùng transform deterministic), rồi chỉ học `Linear(1280, 100)` trên embedding cache.

**Lưu ý công bằng:** probe dùng 40 epoch và không augmentation, các chiến lược khác dùng 10 epoch có augmentation; thời gian của probe **không gồm** thời gian trích xuất đặc trưng.

In [ ]:
def extract_features(pretrained_model: nn.Module, loader: DataLoader) -> TensorDataset:
    """Trích xuất một vector 1280 chiều cho mỗi ảnh; không tính gradient."""
    backbone = copy.deepcopy(pretrained_model)
    backbone.classifier = nn.Identity()
    backbone = backbone.to(DEVICE).eval()
    vectors, labels = [], []
    with torch.inference_mode():
        for images, targets in loader:
            vectors.append(backbone(images.to(DEVICE, non_blocking=True)).cpu())
            labels.append(targets.clone())
    del backbone
    return TensorDataset(torch.cat(vectors), torch.cat(labels))


pretrained_probe = build_efficientnet_b0(pretrained=True)
t0 = time.perf_counter()
cached_train = extract_features(pretrained_probe, train_clean_loader)
cached_val = extract_features(pretrained_probe, val_loader)
print(f"Cached train: {tuple(cached_train.tensors[0].shape)} | Cached val: {tuple(cached_val.tensors[0].shape)}"
      f" | Thời gian trích xuất: {time.perf_counter() - t0:.1f}s")
probe_train_loader = DataLoader(cached_train, batch_size=128, shuffle=True)
probe_val_loader = DataLoader(cached_val, batch_size=256, shuffle=False)

set_seed(SEED)
probe_head = LinearProbe()
probe_head, histories["Linear probe"], summaries["Linear probe"] = fit_model(
    "Linear probe", probe_head, probe_train_loader, probe_val_loader,
    strategy="probe", epochs=EPOCHS_PROBE)

# Ghép head đã học vào backbone pretrained để suy luận trên ảnh.
pretrained_probe.classifier[1].load_state_dict(probe_head.classifier.cpu().state_dict())
fitted_models["Linear probe"] = configure_trainable(pretrained_probe.cpu(), "probe")
summaries["Linear probe"]["Trainable"] = count_parameters(fitted_models["Linear probe"])["Trainable"]
display(pd.DataFrame([summaries["Linear probe"]]).round(4))
del probe_head, cached_train, cached_val, pretrained_probe
torch.cuda.empty_cache()

## 7. Thí nghiệm C — **Partial fine-tuning** (`features[6:]` + head)

Đóng băng `features[0..5]`, mở 3 khối cuối và head. BatchNorm giữ `eval()`. **Discriminative LR:** khối cuối `3e-4`, head `1e-3`.

In [ ]:
set_seed(SEED)
partial_model = configure_trainable(build_efficientnet_b0(pretrained=True), "partial")
print("Số tham số partial:", count_parameters(partial_model))
partial_model, histories["Partial FT"], summaries["Partial FT"] = fit_model(
    "Partial FT", partial_model, train_loader, val_loader, strategy="partial", epochs=EPOCHS_PARTIAL)
fitted_models["Partial FT"] = partial_model.cpu()
display(pd.DataFrame([summaries["Partial FT"]]).round(4))
del partial_model
torch.cuda.empty_cache()

## 8. Thí nghiệm D — **Full fine-tuning** với learning rate theo nhóm khối

Mở toàn mạng với 4 nhóm LR: `features[0:4]` = `5e-5`, `features[4:6]` = `1e-4`, `features[6:9]` = `3e-4`, head = `1e-3`. Các LR này **chưa được tuning**; đây là điểm có thể mở rộng. Full FT khác scratch vì bắt đầu từ pretrained weights.

In [ ]:
set_seed(SEED)
full_model = configure_trainable(build_efficientnet_b0(pretrained=True), "full")
print("Learning rates (full):", [g["lr"] for g in build_optimizer(full_model, "full").param_groups])
full_model, histories["Full FT"], summaries["Full FT"] = fit_model(
    "Full FT", full_model, train_loader, val_loader, strategy="full", epochs=EPOCHS_FULL)
fitted_models["Full FT"] = full_model.cpu()
display(pd.DataFrame([summaries["Full FT"]]).round(4))
del full_model
torch.cuda.empty_cache()

## 9. Tổng hợp và chọn mô hình bằng **validation**

Sắp theo `Val loss` tăng dần (hòa thì `Val macro-F1` giảm dần). **Chỉ mô hình đứng đầu bảng** được đưa ra test. Thời gian của probe chưa gồm trích xuất đặc trưng nên không so sánh trực tiếp chi phí tổng.

In [ ]:
comparison = pd.DataFrame(summaries.values()).sort_values(
    ["Val loss", "Val macro-F1"], ascending=[True, False]).reset_index(drop=True)
display(comparison.round(4))
selected_name = str(comparison.iloc[0]["Mô hình"])
best_model = fitted_models[selected_name].to(DEVICE).eval()
print("Mô hình được chọn CHỈ từ validation:", selected_name)
plot_histories(histories, "Bốn chiến lược trên FGVC-Aircraft: Scratch / Probe / Partial / Full")
comparison.to_csv(OUTPUT_DIR / "validation_comparison.csv", index=False)

In [ ]:
# Tóm tắt tự động từ kết quả vừa chạy (không điền số thủ công)
by_acc = comparison.sort_values("Val accuracy", ascending=False).iloc[0]
by_loss = comparison.iloc[0]
scratch_acc = float(comparison.loc[comparison["Mô hình"] == "Scratch", "Val accuracy"].iloc[0])
probe_acc = float(comparison.loc[comparison["Mô hình"] == "Linear probe", "Val accuracy"].iloc[0])
ft_best_acc = float(comparison.loc[comparison["Mô hình"].isin(["Partial FT", "Full FT"]), "Val accuracy"].max())
print(f"- Val loss thấp nhất: {by_loss['Mô hình']} ({by_loss['Val loss']:.4f}).")
print(f"- Val accuracy cao nhất: {by_acc['Mô hình']} ({by_acc['Val accuracy']:.2%}).")
print(f"- Pretrained (probe) vs Scratch: {probe_acc:.2%} so với {scratch_acc:.2%} "
      f"(chênh {100 * (probe_acc - scratch_acc):+.2f} điểm %).")
print(f"- Fine-tuning tốt nhất vs Linear probe: {ft_best_acc:.2%} so với {probe_acc:.2%} "
      f"(chênh {100 * (ft_best_acc - probe_acc):+.2f} điểm %).")
print("  → Nếu fine-tuning vượt probe rõ rệt, điều này khớp giả thuyết: bài toán fine-grained "
      "cần điều chỉnh đặc trưng, không chỉ dùng đặc trưng ImageNet đóng băng.")

### Hướng dẫn đọc kết quả — So sánh validation
Đọc đồng thời **loss, top-1, macro-F1, top-5**; xếp hạng có thể khác nhau giữa các metric (loss nhạy với độ tự tin của dự đoán). Giới hạn cần ghi rõ: **một seed**, **epoch không bằng nhau** (probe 40, còn lại 10), **chưa tuning siêu tham số**.

## 10. Đánh giá test **đúng một lần**

Test (3.333 ảnh) chỉ dùng **sau khi** đã chọn mô hình bằng validation; không quay lại chỉnh siêu tham số theo test.

In [ ]:
test_loader = DataLoader(aircraft_test, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_metrics = evaluate(best_model, test_loader, NUM_CLASSES)
test_report = pd.DataFrame([{
    "Selected model": selected_name,
    "Test loss": test_metrics["loss"],
    "Test top-1 accuracy": test_metrics["accuracy"],
    "Test macro-F1": test_metrics["macro_f1"],
    "Test top-5 accuracy": test_metrics["top5"],
    "N test": len(aircraft_test),
}])
display(test_report.round(4))
test_report.to_csv(OUTPUT_DIR / "final_test_metrics.csv", index=False)

val_row = comparison.iloc[0]
print(f"Val → Test (cùng mô hình {selected_name}): "
      f"accuracy {val_row['Val accuracy']:.2%} → {test_metrics['accuracy']:.2%} | "
      f"macro-F1 {val_row['Val macro-F1']:.4f} → {test_metrics['macro_f1']:.4f} | "
      f"loss {val_row['Val loss']:.4f} → {test_metrics['loss']:.4f}")

## 11. Phân tích sai số: lớp khó, cặp lớp hay nhầm, ảnh sai tự tin cao

In [ ]:
targets, preds, conf = test_metrics["targets"], test_metrics["preds"], test_metrics["conf"]
cm = confusion_matrix(targets, preds, labels=np.arange(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set(xlabel="Lớp dự đoán (0–99)", ylabel="Lớp thật (0–99)",
       title=f"Confusion matrix — {selected_name} (100 biến thể)")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

report = classification_report(targets, preds, labels=np.arange(NUM_CLASSES),
                               target_names=CLASS_NAMES, zero_division=0, output_dict=True)
class_table = pd.DataFrame(report).T.iloc[:NUM_CLASSES]
print("10 lớp có F1 thấp nhất:")
display(class_table.sort_values("f1-score").head(10).round(3))
class_table.to_csv(OUTPUT_DIR / "test_per_class_metrics.csv")

# Các cặp (lớp thật → lớp dự đoán) bị nhầm nhiều nhất
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off_diag.ravel())[::-1][:10], off_diag.shape))[0]
confusions = pd.DataFrame([{"Lớp thật": CLASS_NAMES[t], "Bị dự đoán thành": CLASS_NAMES[p],
                            "Số ảnh": int(off_diag[t, p])} for t, p in pairs])
print("10 cặp nhầm lẫn nhiều nhất:")
display(confusions)
confusions.to_csv(OUTPUT_DIR / "top_confusions.csv", index=False)

In [ ]:
mistakes = np.where(targets != preds)[0]
print(f"Tổng ảnh dự đoán sai: {len(mistakes)} / {len(aircraft_test)} "
      f"({len(mistakes) / len(aircraft_test):.2%})")

# 8 ảnh sai mà mô hình TỰ TIN nhất
worst = mistakes[np.argsort(conf[mistakes])[::-1][:8]]
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, idx in zip(axes.flat, worst):
    tensor, _ = aircraft_test[int(idx)]
    arr = tensor.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)
    ax.imshow(np.clip(arr, 0, 1))
    ax.set_title(f"Thật: {CLASS_NAMES[targets[idx]]}\nDự đoán: {CLASS_NAMES[preds[idx]]} "
                 f"({conf[idx]:.0%})", fontsize=9)
    ax.axis("off")
fig.suptitle("8 ảnh sai với độ tự tin cao nhất")
plt.tight_layout(); plt.show()

### Hướng dẫn đọc kết quả — Phân tích lỗi
- **Bảng 10 lớp F1 thấp nhất** + **10 cặp nhầm nhiều nhất**: với fine-grained, các cặp thường là hai biến thể cùng dòng (ví dụ cùng họ Boeing 737 hoặc cùng họ Airbus A320) → chỉ khác chi tiết nhỏ.
- **Ảnh sai tự tin cao** cho thấy mô hình không chỉ sai mà còn *quá tự tin*, gợi ý cần hiệu chuẩn (calibration) hoặc dữ liệu/độ phân giải tốt hơn.
- Hãy kiểm tra: lỗi có tập trung trong cùng hãng/họ không? Nếu có, kết quả ở mức `family`/`manufacturer` sẽ cao hơn nhiều so với `variant` (xem phần mở rộng).
- Cần có bằng chứng thêm (nhiều ví dụ, nhãn chính thức) trước khi khẳng định nguyên nhân lỗi.

## 12. Lưu / tải checkpoint và suy luận một ảnh

In [ ]:
checkpoint_path = OUTPUT_DIR / "aircraft_best_efficientnet_b0.pt"
torch.save({
    "architecture": "efficientnet_b0", "num_classes": NUM_CLASSES,
    "annotation_level": ANNOTATION_LEVEL, "class_names": CLASS_NAMES,
    "selected_strategy": selected_name,
    "state_dict": {k: v.detach().cpu() for k, v in best_model.state_dict().items()},
    "image_size": IMAGE_SIZE, "normalization_mean": MEAN, "normalization_std": STD,
    "val_loss": float(comparison.iloc[0]["Val loss"]),
}, checkpoint_path)

payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored_model = build_efficientnet_b0(num_classes=payload["num_classes"], pretrained=False)
restored_model.load_state_dict(payload["state_dict"])
restored_model = restored_model.to(DEVICE).eval()

example_tensor, example_target = aircraft_test[0]
with torch.inference_mode():
    before = best_model.eval()(example_tensor.unsqueeze(0).to(DEVICE)).cpu()
    after = restored_model(example_tensor.unsqueeze(0).to(DEVICE)).cpu()
assert torch.allclose(before, after, atol=1e-4, rtol=1e-4)
values, indices = after.softmax(dim=1)[0].topk(5)
display(pd.DataFrame({"Lớp dự đoán": [payload["class_names"][i] for i in indices.tolist()],
                      "Xác suất": values.tolist()}))
print("Nhãn thật:", CLASS_NAMES[example_target])
print("Checkpoint:", checkpoint_path, "| Kích thước (MB):",
      round(checkpoint_path.stat().st_size / (1024 ** 2), 1))
print("Reload khớp logits:", bool(torch.allclose(before, after, atol=1e-4, rtol=1e-4)))

In [ ]:
# (Tuỳ chọn) tải kết quả về máy
# from google.colab import files
# !zip -qr aircraft_outputs.zip bai_tap_aircraft_outputs
# files.download("aircraft_outputs.zip")

## 13. Nhận xét và kết luận (sinh viên hoàn thiện sau khi chạy)

Điền dựa trên **bảng và hình vừa sinh ra**, không dùng số liệu của Bài 6.

1. **Pretrained vs Scratch:** chênh bao nhiêu điểm % validation accuracy? Vì sao ~33 ảnh/lớp và 10 epoch không đủ cho scratch?
2. **Probe vs Fine-tuning:** fine-tuning có vượt linear probe không? So với kết quả của Flowers-102 (probe đã rất cao), bạn thấy khác biệt gì và giải thích bằng tính chất *fine-grained* / *domain shift* của máy bay.
3. **Loss vs accuracy:** mô hình có loss thấp nhất có trùng mô hình có top-1 cao nhất không? Vì sao tiêu chí chọn mô hình phải công bố trước?
4. **Test vs validation:** khoảng cách bao nhiêu? Có dấu hiệu overfitting vào validation không?
5. **Phân tích lỗi:** nhóm lớp/cặp lớp khó nhất là gì? Nhầm lẫn có tập trung trong cùng hãng/họ?
6. **Giới hạn:** một seed, epoch không đều, LR chưa tuning, probe không augmentation, thời gian probe chưa gồm cache.

### Bài tập mở rộng (tuỳ chọn)
- Chạy lại với `ANNOTATION_LEVEL = "family"` và `"manufacturer"` rồi so sánh độ khó.
- Chạy 3 seed khác nhau và báo cáo trung bình ± độ lệch chuẩn.
- Thử **gradual unfreezing**, **warm-start head từ linear probe**, độ phân giải ảnh lớn hơn (ví dụ 320×320), hoặc `trainval` để huấn luyện lại cuối cùng.
- Thử `label_smoothing=0.1` hoặc temperature scaling để giảm quá tự tin.